# Step 4.5: the executor

This notebook is the check of step 4.5. FEAT now runs your code, a feature's preview or a snippet of Python, in a process of its own, a runner, within a time limit and a memory limit, so that a loop that never ends, or a memory explosion, stops there rather than taking the app down. Here you give the executor a snippet that sleeps for 30 seconds under a 5-second timeout.

Run All from the top. The notebook reads none of your market data, and writes only in `FEFAT_HOME/cache/executions`, which the executor empties after each run.

## A snippet that runs past its timeout

The executor starts a runner, which reports that it runs the snippet, then stops it, with every process it started, once the 5 seconds are up.

Expected: `30% Running the snippet`, then `timed_out` after about 5 seconds.

In [ ]:
import os
import time

import psutil

from fefat.config import resolve_paths
from fefat_app.execution import ExecutionLimits, PythonSnippetSpec
from fefat_app.executor.subprocess_executor import SubprocessExecutor

paths = resolve_paths(os.environ)
executor = SubprocessExecutor(paths, warm_previews=False)
sleeps = PythonSnippetSpec(
    code="import time\ntime.sleep(30)\n",
    bars=None,
    limits=ExecutionLimits(timeout_seconds=5, memory_bytes=2 * 1024**3),
)
began = time.monotonic()
result = executor.run(sleeps, lambda progress: print(f"{progress.percent:.0f}% {progress.message}"))
took = time.monotonic() - began
print(f"{result.outcome} after {took:.1f} s")

## Nothing left behind

The runner ended with the snippet, and the executor removed the snippet's folder and the runner's log.

Expected: `PASSED` twice.

In [ ]:
runners = [
    child for child in psutil.Process().children() if "fefat_app.executor" in child.cmdline()
]
left = sorted(paths.executions_dir.glob("*"))
print(f"No runner still runs: {'PASSED' if not runners else 'FAILED'}")
print(f"Nothing left in the executions' folder: {'PASSED' if not left else 'FAILED'}")